# 04 - Fine-Tuning de ConvNeXt para Classificacao de Especies com Ruido Artificial (5%)

Executa uma adaptação auditável por ruído em fragmentos de 512 bp. Esta execução identificada não demonstra equivalência aos checkpoints publicados nem reproduz, por si só, as Tabelas 1, 3 e 4.
- Treinamento com corrupcao artificial de sequencias (epsilon = 0.05) aplicada em treino e validacao (AD-009, Notas 21-23).
- Carrega o checkpoint de especie base (`convnext_transfer_species`) com 13.683 classes (AD-001).
- Preserva o modo histórico observado e registra divergências de configuração:
  1. `MODE = "historical_reproduction"`: hiperparametros historicos (`dropout=0.7`, `lr=5e-5`, `AdamW`, `CosineAnnealingLR`, `label_smoothing=0.1`, selecao de melhor validacao na epoca 7 `finetune_noise_7.pt`).
  2. A comparação controlada do AD-016 permanece `NOT_RUN` nesta trilha mínima.
- Desacoplamento via `src/viral_cnn/` sem `import *` e salvamento atômico em namespace exclusivo da execução sem alterar os pesos historicos (AGENTS.md).


In [4]:
import os
import sys
import uuid
from dataclasses import asdict
from pathlib import Path

# Resolucao autonoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import torch
import pandas as pd

from viral_cnn.notebook_runtime import (
    setup_notebook_runtime,
    ensure_smoke_test_artifacts,
    write_notebook_manifest,
    resolve_split_files,
    create_optimized_dataloader,
    reserve_run_directory,
)
from viral_cnn.models.registry import REGISTERED_VARIANTS, build_registered_model
from viral_cnn.data.parquet_dataset import ParquetBatchIterableDataset
from viral_cnn.training.audit import (
    protocol_audit,
    require_empirical_inputs,
    select_checkpoint_epoch,
    sha256_file,
    taxonomy_order_sha256,
    write_c05_manifest,
)
from viral_cnn.training.config import ControlledExperimentConfig
from viral_cnn.training.trainer import ControlledTrainer

ctx = setup_notebook_runtime()
print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")

MODE = "historical_reproduction"
NOISE_RATE = 0.05
EPOCHS = 1 if ctx.is_smoke_test else 15
MAX_BATCHES = 2 if ctx.is_smoke_test else None
RUN_ID = os.getenv("RUN_ID") or f"c05-{uuid.uuid4().hex[:12]}"
RESUME_CHECKPOINT = os.getenv("RESUME_CHECKPOINT", "").strip()


Device: cuda
Smoke Test Mode: False
Data Root: /home/mlima/workspace


In [5]:
# 2. Reserve isolated outputs and validate all empirical inputs before model construction
if ctx.is_smoke_test:
    ensure_smoke_test_artifacts(ctx.data_root)

project_root = next(
    (root for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (root / "src" / "viral_cnn").exists()),
    Path.cwd().resolve(),
)
run_dir = reserve_run_directory(project_root, RUN_ID, "convnext_finetune_species")
output_dir = run_dir / "outputs"
checkpoint_dir = run_dir / "checkpoints"
output_dir.mkdir(parents=True)
checkpoint_dir.mkdir(parents=True)

base_model_name = "convnext_transfer_species"
species_parquet_dir = Path(ctx.topology.paths["short"]["species_parquet_dir"])
taxonomy_path = Path(ctx.topology.paths["short"]["species_csv"])
train_files = resolve_split_files(species_parquet_dir, "train")
val_files = resolve_split_files(species_parquet_dir, "validation")
test_files = resolve_split_files(species_parquet_dir, "test")
base_checkpoint = Path(ctx.topology.paths["checkpoints"][base_model_name]).resolve()

if ctx.is_smoke_test:
    input_identities = {
        "train": [{"path": str(Path(p).resolve()), "sha256": sha256_file(p)} for p in train_files],
        "validation": [{"path": str(Path(p).resolve()), "sha256": sha256_file(p)} for p in val_files],
        "test": [{"path": str(Path(p).resolve()), "sha256": sha256_file(p)} for p in test_files],
        "taxonomy_mapping": [{"path": str(taxonomy_path.resolve()), "sha256": sha256_file(taxonomy_path)}],
        "base_checkpoint": [{"path": str(base_checkpoint), "sha256": "unavailable-synthetic-initialization"}],
    }
else:
    input_identities = require_empirical_inputs(
        train_files, val_files, test_files, taxonomy_path, base_checkpoint,
    )

class_order_sha, class_count = taxonomy_order_sha256(taxonomy_path)
if not ctx.is_smoke_test and class_count != 13683:
    raise ValueError(f"Expected 13683 Species classes in canonical order, found {class_count}")
if not ctx.is_smoke_test and (len(train_files) == 0 or len(val_files) == 0 or len(test_files) == 0):
    raise FileNotFoundError("Empirical C05 requires train, validation, and test partitions")

train_ds = ParquetBatchIterableDataset(
    train_files, batch_size=32 if ctx.is_smoke_test else 4096,
    shuffle=True, seed=ctx.seed, split="train",
)
train_loader = create_optimized_dataloader(train_ds, is_smoke_test=ctx.is_smoke_test, device=ctx.device)
val_ds = ParquetBatchIterableDataset(
    val_files, batch_size=32 if ctx.is_smoke_test else 4096,
    shuffle=False, seed=ctx.seed, split="validation",
)
val_loader = create_optimized_dataloader(val_ds, is_smoke_test=ctx.is_smoke_test, device=ctx.device)
print(f"[C05] run_id={RUN_ID} train={len(train_files)} validation={len(val_files)} test={len(test_files)}")


[LOG_PATH] Loading DataLoader/ParquetDataset from: /home/mlima/workspace/cnn_kraken/processed_kraken_species_splited_parquet/train/part.0.parquet ... (256 files total) | 256 partitions, 13650160 samples, batch_size=4096


DataLoaders ready. Train files: 256, Val files: 256


[LOG_PATH] Loading DataLoader/ParquetDataset from: /home/mlima/workspace/cnn_kraken/processed_kraken_species_splited_parquet/validation/part.0.parquet ... (256 files total) | 256 partitions, 1706271 samples, batch_size=4096


In [6]:
# 3. Load the identified base checkpoint and freeze the effective protocol
model = build_registered_model(
    base_model_name, checkpoint_path=None if ctx.is_smoke_test else base_checkpoint,
    data_root=ctx.data_root, device=ctx.device,
)
if hasattr(torch, "compile") and ctx.device.type == "cuda":
    try:
        model = torch.compile(model, dynamic=True)
        print("Model compiled with torch.compile(dynamic=True)")
    except Exception as exc:
        print(f"torch.compile skipped: {exc}")

config = ControlledExperimentConfig(
    architectures=("convnext_tiny_1d",), num_classes=class_count,
    learning_rate=5e-5, weight_decay=1e-4, warmup_steps=0,
    label_smoothing=0.1, dropout_rate=0.7, noise_rate=NOISE_RATE,
    checkpoint_metric="last_step" if "best_val_macro_f1" == "last" else "best_val_macro_f1",
    mode="historical_reproduction", scheduler_step_unit="epoch",
)
trainer = ControlledTrainer(model=model, config=config, device=ctx.device)
scheduler_t_max = max(1, EPOCHS * len(train_ds))
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    trainer.optimizer, T_max=scheduler_t_max, eta_min=1e-7,
)
noise_generator = torch.Generator(device="cpu").manual_seed(ctx.seed)
model_dropout = [float(module.p) for module in getattr(model, "_orig_mod", model).modules() if isinstance(module, (torch.nn.Dropout, torch.nn.Dropout1d, torch.nn.Dropout2d))]
effective_dropout = model_dropout[-1] if model_dropout else None
protocol = protocol_audit(
    base_variant=base_model_name,
    base_checkpoint_sha256=input_identities["base_checkpoint"][0]["sha256"],
    rank="species", num_classes=class_count, class_order_sha256=class_order_sha,
    effective_dropout=effective_dropout,
    dropout_source="loaded model modules" if effective_dropout is not None else "unknown",
    config={**asdict(config), "epochs": EPOCHS, "scheduler": "CosineAnnealingLR", "scheduler_step_unit": "epoch", "scheduler_t_max": scheduler_t_max, "scheduler_eta_min": 1e-7, "selection_rule": "best_val_macro_f1"},
)
run_identity = {
    "protocol": "c05_finetune_audit_v1", "base_variant": base_model_name,
    "base_checkpoint_sha256": protocol["base_checkpoint_sha256"],
    "class_order_sha256": class_order_sha, "input_sha256": input_identities,
    "configuration": protocol["configuration"],
}

print("--- Pre-training Evaluation (Epoch 0) --- [limited preview]")
preview_limit = 2 if ctx.is_smoke_test else 10
trainer.config.noise_rate = 0.0
val_clean_pre = trainer.evaluate_epoch(val_loader, max_batches=preview_limit, generator=noise_generator)
trainer.config.noise_rate = 0.05
val_noise_pre = trainer.evaluate_epoch(val_loader, max_batches=preview_limit, generator=noise_generator)
trainer.config.noise_rate = NOISE_RATE
pretraining_preview = {
    "split": "validation", "support_clean": val_clean_pre["total_samples"],
    "support_epsilon_0_05": val_noise_pre["total_samples"], "max_batches": preview_limit,
    "complete_split": not ctx.is_smoke_test and preview_limit is None,
    "epsilon_clean": 0.0, "epsilon_noise": 0.05,
    "clean": val_clean_pre, "epsilon_0_05": val_noise_pre,
}
print(f"Epoch 0 Pre-Train Val Clean: Acc={val_clean_pre['val_acc']:.4f} | Macro-F1={val_clean_pre['val_f1_macro']:.4f} | Support={val_clean_pre['total_samples']}")
print(f"Epoch 0 Pre-Train Val Noise (epsilon=0.05): Acc={val_noise_pre['val_acc']:.4f} | Macro-F1={val_noise_pre['val_f1_macro']:.4f} | Support={val_noise_pre['total_samples']} | Preview batches={preview_limit}")


Loading base model: convnext_transfer_species


[LOG_PATH] Loading checkpoint from: /home/mlima/workspace/cnn_kraken/checkpoint_transfer_species_conv/best_convnext_classifier_fast_phase4_2_restarts.pt


Model compiled with torch.compile(dynamic=True)
Trainer and CosineAnnealingLR configured.
--- Pre-training Evaluation (Epoch 0) ---
Epoch 0 Pre-Train Val Clean (eps=0.0): Acc=0.5223 | Macro-F1=0.4660
Epoch 0 Pre-Train Val Noise (eps=0.05): Acc=0.3110 | Macro-F1=0.3067


In [7]:
# 4. Noise-aware fine-tuning; a resume is opt-in and must match the frozen run identity
training_history = []
best_val_f1 = None
start_epoch = 1
if RESUME_CHECKPOINT:
    print(f"[LOG_PATH] Explicit resume requested from: {Path(RESUME_CHECKPOINT).resolve()}")
    start_epoch, training_history = trainer.load_resume_checkpoint(
        RESUME_CHECKPOINT, scheduler=lr_scheduler, expected_identity=run_identity,
        noise_generator=noise_generator, require_complete_state=True,
    )
    best_val_f1 = trainer.best_metric_value
    print(f"Resumed from Epoch {{start_epoch}} with {{len(training_history)}} history records. Best Val Macro-F1: {{best_val_f1}}")

completed_epochs = start_epoch - 1
run_status = "RUNNING"
try:
    if pretraining_preview["support_clean"] == 0 or pretraining_preview["support_epsilon_0_05"] == 0:
        raise RuntimeError("Validation preview had zero observed support; refusing empirical fine-tuning")
    for epoch in range(start_epoch, EPOCHS + 1):
        train_res = trainer.train_epoch(
            train_loader, max_batches=MAX_BATCHES, generator=noise_generator,
            scheduler=lr_scheduler, scheduler_step_unit="epoch",
        )
        val_res = trainer.evaluate_epoch(val_loader, max_batches=MAX_BATCHES, generator=noise_generator)
        if train_res["total_samples"] == 0 or val_res["total_samples"] == 0:
            raise RuntimeError("Training or validation epoch had zero observed support")
        record = {
            "epoch": epoch, "train_loss": train_res["train_loss"],
            "train_acc": train_res["train_acc"], "train_top5": train_res.get("train_acc_top5"),
            "train_f1_macro": train_res["train_f1"], "train_f1_weighted": train_res["train_f1_weighted"],
            "train_support": train_res["total_samples"], "train_steps_attempted": train_res["steps_attempted"],
            "train_steps_effective": train_res["steps_effective"],
            "val_loss": val_res["val_loss"], "val_acc": val_res["val_acc"],
            "val_top5": val_res["val_acc_top5"], "val_f1_macro": val_res["val_f1_macro"],
            "val_f1_weighted": val_res["val_f1_weighted"], "val_support": val_res["total_samples"],
            "learning_rate": train_res["learning_rate"], "scheduler_step_unit": "epoch",
            "noise_epsilon": 0.05, "noise_seed": ctx.seed,
        }
        training_history.append(record)
        trainer.training_history = training_history
        print(f"Epoch {epoch}/{EPOCHS} train={train_res['train_loss']:.4f} val={val_res['val_loss']:.4f} Macro-F1={val_res['val_f1_macro']:.4f} Top-5={val_res['val_acc_top5']:.4f} LR={train_res['learning_rate']:.3e} steps={train_res['steps_effective']}/{train_res['steps_attempted']}")

        score = float(val_res["val_f1_macro"])
        is_best = best_val_f1 is None or score > best_val_f1
        if is_best:
            best_val_f1 = score
            trainer.best_metric_value = score
        last_checkpoint = checkpoint_dir / "finetune_noise_last.pt"
        trainer.save_epoch_checkpoint(
            epoch, record, last_checkpoint, scheduler=lr_scheduler,
            training_history=training_history, run_identity=run_identity, noise_generator=noise_generator,
            checkpoint_role="last",
        )
        if is_best:
            trainer.save_epoch_checkpoint(
                epoch, record, checkpoint_dir / "finetune_noise_best.pt", scheduler=lr_scheduler,
                training_history=training_history, run_identity=run_identity, noise_generator=noise_generator,
                checkpoint_role="best",
            )
        completed_epochs = epoch
    run_status = "COMPLETE" if completed_epochs >= EPOCHS else "PARTIAL"
except Exception as exc:
    run_status = "FAILED"
    failed_outputs = [p for p in (checkpoint_dir / "finetune_noise_last.pt", checkpoint_dir / "finetune_noise_best.pt") if p.is_file()]
    write_c05_manifest(
        output_dir / "c05_manifest.json", protocol=protocol, status="FAILED",
        is_smoke_test=ctx.is_smoke_test, inputs=input_identities, outputs=failed_outputs,
        completed_epochs=completed_epochs, planned_epochs=EPOCHS, selected_checkpoint=None,
        metrics={"completed_history": training_history, "failure": f"{type(exc).__name__}: {exc}"},
        execution={"run_id": RUN_ID, "notebook_id": "convnext_finetune_species", "seed": ctx.seed, "device": str(ctx.device), "resume_source": str(Path(RESUME_CHECKPOINT).resolve()) if RESUME_CHECKPOINT else None},
    )
    raise


--- Starting convnext_tiny_1d Fine-Tuning (Noise Rate: 5.0%, Epochs: 15, Mode: historical_reproduction) ---
Found existing checkpoint at results_reproduction/checkpoints/convnext_noise/finetune_noise_last.pt. Resuming training...
Resumed from Epoch 4 with 3 history records. Best Val Macro-F1: 0.4129
Epoch 4/15 (3465.6s) -> Train Loss: 3.3631 | Train Acc: 0.5894 | Val Loss: 5.0356 | Val Acc: 0.4022 | Val Macro-F1: 0.4180 | LR: 5.00e-05
Epoch 5/15 (2392.4s) -> Train Loss: 3.4640 | Train Acc: 0.5608 | Val Loss: 5.0951 | Val Acc: 0.3948 | Val Macro-F1: 0.4103 | LR: 5.00e-05
Epoch 6/15 (2399.0s) -> Train Loss: 3.5324 | Train Acc: 0.5414 | Val Loss: 5.1890 | Val Acc: 0.3827 | Val Macro-F1: 0.3988 | LR: 5.00e-05
Epoch 7/15 (2396.1s) -> Train Loss: 3.5113 | Train Acc: 0.5455 | Val Loss: 5.2049 | Val Acc: 0.3806 | Val Macro-F1: 0.3963 | LR: 5.00e-05
Epoch 8/15 (2398.9s) -> Train Loss: 3.4934 | Train Acc: 0.5489 | Val Loss: 5.1928 | Val Acc: 0.3850 | Val Macro-F1: 0.3998 | LR: 5.00e-05
Epoch 9/1

In [11]:
# 5. Select on validation, evaluate the frozen selection on the identified test split, and write evidence
history_df = pd.DataFrame(training_history)
print("=== Training History ===")
print(history_df.to_string())
history_csv = output_dir / "training_history.csv"
history_json = output_dir / "training_history.json"
history_df.to_csv(history_csv, index=False)
history_df.to_json(history_json, orient="records", indent=2)

selection_rule = "best_val_macro_f1"
selected_epoch = select_checkpoint_epoch(training_history, selection_rule) if training_history else None
selected_role = "last" if selection_rule == "last" else "best"
role_filenames = {"last": "finetune_noise_last.pt", "best": "finetune_noise_best.pt"}
def resolve_role_checkpoint(role):
    local_path = checkpoint_dir / role_filenames[role]
    if local_path.is_file():
        return local_path
    if RESUME_CHECKPOINT:
        source_path = Path(RESUME_CHECKPOINT).resolve().parent / role_filenames[role]
        if source_path.is_file():
            source_payload = torch.load(source_path, map_location="cpu", weights_only=False)
            if source_payload.get("checkpoint_role") == role and source_payload.get("run_identity") == run_identity:
                return source_path
    return local_path
selected_path = resolve_role_checkpoint(selected_role)
selected_record = None
if selected_epoch is not None and selected_path.is_file():
    selected_payload = torch.load(selected_path, map_location="cpu", weights_only=False)
    target_model = getattr(trainer.model, "_orig_mod", trainer.model)
    target_model.load_state_dict(selected_payload["model_state_dict"])
    selected_record = {
        "role": "selected", "path": str(selected_path.resolve()),
        "sha256": sha256_file(selected_path), "rule": selection_rule,
        "epoch": selected_epoch, "metric": "val_f1_macro" if selection_rule != "last" else None,
        "value": best_val_f1 if selection_rule != "last" else None,
    }

# Test rows and loader are instantiated only after selection is frozen.
test_results = {}
if selected_record is not None:
    test_ds = ParquetBatchIterableDataset(
        test_files, batch_size=32 if ctx.is_smoke_test else 4096,
        shuffle=False, seed=ctx.seed, split="test",
    )
    test_loader = create_optimized_dataloader(test_ds, is_smoke_test=ctx.is_smoke_test, device=ctx.device)
    for epsilon in (0.0, 0.02, 0.05, 0.10, 0.15):
        trainer.config.noise_rate = epsilon
        test_metrics = trainer.evaluate_epoch(
            test_loader, max_batches=MAX_BATCHES,
            generator=torch.Generator(device="cpu").manual_seed(ctx.seed + int(epsilon * 1000)),
        )
        test_results[f"epsilon_{epsilon:.2f}"] = {
            **test_metrics, "epsilon": epsilon,
            "support": test_metrics["total_samples"],
            "seed": ctx.seed + int(epsilon * 1000),
            "complete_split": MAX_BATCHES is None,
        }
trainer.config.noise_rate = NOISE_RATE
if not ctx.is_smoke_test and (not test_results or any(result["total_samples"] == 0 or not result["complete_split"] for result in test_results.values())):
    run_status = "PARTIAL"

checkpoint_roles = {}
for role in ("last", "best"):
    checkpoint_path = resolve_role_checkpoint(role)
    checkpoint_roles[role] = {"path": str(checkpoint_path.resolve()), "sha256": sha256_file(checkpoint_path) if checkpoint_path.is_file() else None, "checkpoint_role": role if checkpoint_path.is_file() else None}
checkpoint_roles["selected"] = selected_record
manifest = write_c05_manifest(
    output_dir / "c05_manifest.json", protocol=protocol, status=run_status,
    is_smoke_test=ctx.is_smoke_test, inputs=input_identities,
    outputs=[history_csv, history_json, *[Path(role["path"]) for role in checkpoint_roles.values() if role.get("sha256")]],
    completed_epochs=completed_epochs, planned_epochs=EPOCHS,
    selected_checkpoint=selected_record, checkpoint_roles=checkpoint_roles,
    metrics={"pretraining_preview": pretraining_preview, "final_epoch": training_history[-1] if training_history else {}, "test": test_results},
    execution={
        "run_id": RUN_ID, "notebook_id": "convnext_finetune_species", "seed": ctx.seed,
        "device": str(ctx.device), "data_root": str(ctx.data_root),
        "train_files": train_files, "validation_files": val_files, "test_files": test_files,
        "resume_source": str(Path(RESUME_CHECKPOINT).resolve()) if RESUME_CHECKPOINT else None,
        "command": "Jupyter notebook 03/04 C05 fine-tuning audit",
        "selection_uses_test": False,
        "noise_operation_order": ["pretraining_clean", "pretraining_validation_epsilon_0.05", "training_epsilon_0.05", "validation_epsilon_0.05", "test_noise_sweep"],
        "train_workers": train_loader.num_workers, "validation_workers": val_loader.num_workers, "test_workers": test_loader.num_workers if selected_record is not None else None,
        "determinism_limits": "PyTorch worker scheduling and CUDA kernel determinism are not proven by local smoke runs",
        "interpretation": "single fine-tuning configuration; no architecture-only causal claim",
    },
)
standard_manifest = write_notebook_manifest(
    notebook_id="convnext_finetune_species",
    data_root=ctx.data_root, is_smoke_test=ctx.is_smoke_test,
    metrics_summary={"c05_status": manifest["status"], "completed_epochs": completed_epochs, "planned_epochs": EPOCHS, "test": test_results},
    checkpoints={base_model_name: str(base_checkpoint), "selected_checkpoint": str(selected_path) if selected_record else "unknown"},
    input_files=[*train_files, *val_files, *test_files, str(taxonomy_path), str(base_checkpoint)],
    seed=ctx.seed, output_json_path=output_dir / "manifest.json",
    output_yaml_path=output_dir / "manifest.yaml", run_id=RUN_ID,
    configuration={"protocol": protocol, "run_identity": run_identity, "selection_rule": selection_rule, "run_status": manifest["status"]},
    outputs=[history_csv, history_json, *[Path(role["path"]) for role in checkpoint_roles.values() if role.get("sha256")]],
    command="Jupyter reproduction notebook C05 fine-tuning audit",
)
print(f"C05 status={manifest['status']}; manifest={output_dir / 'c05_manifest.json'}")


=== Training History ===
    epoch  train_loss  train_acc  train_f1  train_f1_weighted  val_loss   val_acc  val_f1_macro  val_f1_weighted  learning_rate  epoch_seconds
0       1    3.645575   0.521640  0.514328           0.514364  5.166459  0.386418      0.404795         0.404888        0.00005        3474.73
1       2    3.588136   0.531412  0.522910           0.522946  5.143893  0.388406      0.405848         0.405929        0.00005        2397.02
2       3    3.469418   0.560731  0.551673           0.551709  5.078570  0.396496      0.412947         0.413029        0.00005        2391.91
3       4    3.363114   0.589408  0.580171           0.580204  5.035608  0.402186      0.418029         0.418121        0.00005        3465.61
4       5    3.463968   0.560793  0.551727           0.551761  5.095107  0.394831      0.410328         0.410429        0.00005        2392.37
5       6    3.532433   0.541354  0.532417           0.532454  5.188961  0.382739      0.398791         0.398814     